# Análise do Sistema de Informação sobre Mortalidade – SIM

### 1. Origem dos dados, suas características e recorte temporal.

Os dados foram obtidos por meio do **Portal Brasileiro de Dados Abertos do Governo Federal**. A base é desenvolvida, consolidada e coordenada pelo **Ministério da Saúde**.
O arquivo **Mortalidade_Geral_2026.csv** consiste em um conjunto de dados públicos estruturado que reúne os registros oficiais de óbitos em território nacional. Ele possui **86 colunas**, cujas descrições e características constam no dicionário de dados **(Dicionario_SIM_2025.pdf)**, também extraído do **Portal Brasileiro de Dados Abertos do Governo Federal**.
A referida base de dados foi delimitada ao **período de 01/01/2026 a 31/05/2026**, permitindo uma análise focada no desempenho do primeiro semestre do ano de 2026, totalizando **506531 linhas**.

### Perguntas a serem respondidas:

1. Quantidade de mortes por mês?
2. Quantidade de mortes por estado?
3. Quantidade de mortes por faixa etária?
4. As 5 maiores causas de morte(CID)?
5. As 5 cidades com maior número de óbitos? 

### 2. Diagnóstico de qualidade

2.1 - Dimensões, tipos, uso de memória.

In [41]:
import pandas as pd
import numpy as np

mortalidade_completa = pd.read_csv(
    '../dados/Mortalidade_Geral_2026.csv',
    sep=';',
    parse_dates=['DTOBITO', 'DTNASC'],
    date_format='%d/%m/%Y',
    dtype={'CAUSAMAT': str}
)

# Dimensões
print("Dimensões (Linhas, Colunas):", mortalidade_completa.shape)
# Tipos
print("\nTipos (Colunas, Tipos):")
print(mortalidade_completa.dtypes)
# Uso de Memória
print("\nMemória Utilizada:")
mortalidade_completa.info(verbose=False, memory_usage='deep')


Dimensões (Linhas, Colunas): (506531, 86)

Tipos (Colunas, Tipos):
contador        int64
ORIGEM          int64
TIPOBITO        int64
DTOBITO           str
HORAOBITO     float64
               ...   
ALTCAUSA      float64
CAUSABAS_O        str
TPPOS             str
TP_ALTERA     float64
CB_ALT            str
Length: 86, dtype: object

Memória Utilizada:
<class 'pandas.DataFrame'>
RangeIndex: 506531 entries, 0 to 506530
Columns: 86 entries, contador to CB_ALT
dtypes: float64(56), int64(12), str(18)
memory usage: 667.1 MB


2.2 - Faltantes por coluna (quantidade e percentual).

In [42]:
# Quantidade absoluta de dados faltantes por coluna
print("--- Quantidade de Faltantes ---")
print(mortalidade_completa.isnull().sum())

# Percentual de dados faltantes por coluna (arredondado em 2 casas decimais)
print("\n--- Percentual (%) de Faltantes ---")
print((mortalidade_completa.isnull().mean() * 100).round(2))


--- Quantidade de Faltantes ---
contador           0
ORIGEM             0
TIPOBITO           0
DTOBITO            0
HORAOBITO      14739
               ...  
ALTCAUSA      502188
CAUSABAS_O       990
TPPOS         173848
TP_ALTERA     496208
CB_ALT        504104
Length: 86, dtype: int64

--- Percentual (%) de Faltantes ---
contador       0.00
ORIGEM         0.00
TIPOBITO       0.00
DTOBITO        0.00
HORAOBITO      2.91
              ...  
ALTCAUSA      99.14
CAUSABAS_O     0.20
TPPOS         34.32
TP_ALTERA     97.96
CB_ALT        99.52
Length: 86, dtype: float64


2.3 - Duplicados, categorias inconsistentes, valores inválidos.

In [43]:
# Quantidade total de linhas 100% duplicadas na tabela inteira
total_duplicados = mortalidade_completa.duplicated().sum()
print(f"Total de linhas duplicadas: {total_duplicados}")
mortalidade_completa[mortalidade_completa.duplicated(keep=False)]

# Categorias Inconsistentes
print("\nColuna RACACOR:")
mortalidade_completa['RACACOR'].value_counts(dropna=False)

Total de linhas duplicadas: 0

Coluna RACACOR:


RACACOR
1.0    250465
4.0    198952
2.0     45706
NaN      6245
3.0      3246
5.0      1917
Name: count, dtype: int64

2.4 - Identificação de outliers com método justificado (IQR ou z-score), avaliados no **grupo de comparação correto**

**Justificativa:** O Método IQR (Intervalo Interquartil) foi escolhido por ser o mais adequado para analisar dados de idade de óbitos, que possuem uma distribuição severamente assimétrica (com grande concentração de registros na faixa dos idosos). Diferente de outros métodos (como o Z-score), o IQR não exige que os dados sigam uma distribuição normal (simétrica) e é altamente robusto, evitando que a média geral seja distorcida por valores extremos. A análise foi aplicada de forma unidirecional (apenas na cauda superior) e segmentada por grupo de comparação (Sexo). Essa abordagem foi necessária porque os limites inferiores matemáticos do IQR classificavam mortes prematuras reais (como crianças e jovens) como "anomalias". Focando apenas no limite superior, o método isolou com precisão cirúrgica apenas os outliers biológicos legítimos (supercentenários), cujas idades destoam completamente do padrão de longevidade da população ativa.

In [65]:
# Separar a IDADE em Tipo (1º dígito) e Quantidade (2º e 3º dígitos)
mortalidade_completa['IDADELIMPA'] = ("000" + mortalidade_completa['IDADE'].astype(str)).str[-3:]

tipo_idade = mortalidade_completa['IDADELIMPA'].str[0]
qtde_idade = pd.to_numeric(mortalidade_completa['IDADELIMPA'].str[1:], errors='coerce')

qtde_idade.loc[qtde_idade == 0] = 1

# 2. Aplicar a regra de conversão para ANOS
mortalidade_completa['IDADEANOS'] = np.select(
    [
        tipo_idade == '0',  # Minutos
        tipo_idade == '1',  # Horas
        tipo_idade == '2',  # Dias
        tipo_idade == '3',  # Meses
        tipo_idade == '4',  # Anos (< 100)
        tipo_idade == '5'   # Anos (>= 100)
    ],
    [
        qtde_idade / (60 * 24 * 365),
        qtde_idade / (24 * 365),
        qtde_idade / 365,
        qtde_idade / 12,
        qtde_idade,
        qtde_idade + 100
    ],
    default=np.nan  # Idades ignoradas ou nulas
)

mortalidade_completa.loc[mortalidade_completa['IDADEANOS'] < 1, 'IDADEANOS'] = 0

# Resumo estatístico das colunas numéricas (atenção para os valores 'min' e 'max')
mortalidade_completa.describe()

# Função para capturar estritamente os outliers superiores
def encontrar_outliers_reais(grupo):
    idades_validas = grupo['IDADEANOS'].dropna()
    Q1 = idades_validas.quantile(0.25)
    Q3 = idades_validas.quantile(0.75)
    IQR = Q3 - Q1
    
    limite_superior = Q3 + (1.5 * IQR)
    
    # Retorna True apenas para quem passou do teto estatístico do grupo
    return grupo['IDADEANOS'] > limite_superior

# Filtra a base utilizando a nova máscara
outliers_superiores_mascara = mortalidade_completa.groupby('SEXO', group_keys=False).apply(encontrar_outliers_reais)
# df_outliers_finais = mortalidade_completa[outliers_superiores_mascara]
df_outliers_finais = mortalidade_completa[outliers_superiores_mascara.reindex(mortalidade_completa.index, fill_value=False)]

# Exibe o veredito final
print(f"Total de outliers superiores encontrados: {len(df_outliers_finais)}")

# Exibe as colunas principais dos 6 outliers encontrados
df_outliers_finais[['SEXO', 'IDADEANOS', 'DTOBITO', 'CAUSAMAT']]



Total de outliers superiores encontrados: 6


,SEXO,IDADEANOS,DTOBITO,CAUSAMAT
158675,1,122.0,23012026,NaN
163467,2,122.0,19032026,NaN
202230,2,120.0,25042026,NaN
228759,2,121.0,26032026,NaN
287885,1,126.0,30032026,NaN
469799,1,125.0,10012026,NaN


### 3. Limpeza e transformação

1. Tratamento de Dados Faltantes (Missing Values)Decisão de Tratamento:Colunas Categóricas (SEXO, RACACOR): Imputação de valor fixo textual (ex: "Não informado").Justificativa: Remover as linhas descartaria dados epidemiológicos valiosos de outras colunas. Imputar pela categoria mais frequente (moda) geraria um viés artificial. Manter a classificação como "Não informado" preserva a integridade do banco e reflete falhas reais de preenchimento na ponta do sistema de saúde.Colunas Numéricas/Temporais (IDADEANOS): Para as linhas restantes com idade nula, aplicar a imputação pela mediana do grupo de comparação específico (ex: agrupado por CAUSAMAT ou SEXO) em vez da média geral.Justificativa: A média geral é altamente sensível a extremos e distorce a realidade biológica (a expectativa de vida muda drasticamente dependendo do sexo e da causa da morte). A mediana por grupo garante consistência biológica e não é afetada por assimetrias na distribuição.

2. Remoção e Consolidação de DuplicadosDecisão de Tratamento:Duplicados Totais: Eliminação estrita de linhas 100% idênticas mantendo a primeira ocorrência (keep='first').Justificativa: No registro de óbitos, duas notificações idênticas em todos os campos (mesmo ID/Declaração, data, idade, sintomas) caracterizam erro de duplicidade de envio ou processamento do arquivo de dados.

In [72]:
# Categóricas: Tratando faltantes com categoria explícita
mortalidade_completa['SEXO'] = mortalidade_completa['SEXO'].fillna('Não informado')
mortalidade_completa['RACACOR'] = mortalidade_completa['RACACOR'].fillna('Não informado')

# Numéricas: Imputação pela mediana do grupo de Sexo
mediana_por_sexo = mortalidade_completa.groupby('SEXO')['IDADEANOS'].transform('median')
mortalidade_completa['IDADEANOS'] = mortalidade_completa['IDADEANOS'].fillna(mediana_por_sexo)

# Remove duplicados idênticos mantendo o primeiro registro
linhas_antes = lenmortalidade_completa = len(mortalidade_completa)
mortalidade_completa = mortalidade_completa.drop_duplicates(keep='first')
print(f"Registros duplicados removidos: {linhas_antes - len(mortalidade_completa)}")

# Padronização de Raça/Cor usando .map()
dicionario_raca = {
    1: 'Branca',
    2: 'Preta',
    3: 'Amarela',
    4: 'Parda',
    5: 'Indígena'
}
mortalidade_completa['RACACOR'] = mortalidade_completa['RACACOR'].map(dicionario_raca).fillna('Não informado')

# Remoção de espaços invisíveis usando .str.strip() nas strings
mortalidade_completa['CAUSAMAT'] = mortalidade_completa['CAUSAMAT'].astype(str).str.strip().str.upper()

# Define as quebras (bins) e as etiquetas correspondentes
bins_idade = [-1, 1, 12, 19, 59, 150]
labels_idade = ['Infantil (Até 1 ano)', 'Criança (2-12)', 'Adolescente (13-19)', 'Adulto (20-59)', 'Idoso (60+)']

mortalidade_completa['FAIXA_ETARIA'] = pd.cut(
    mortalidade_completa['IDADEANOS'], 
    bins=bins_idade, 
    labels=labels_idade
)

# Classifica de forma binária com base na idade limite
mortalidade_completa['OBITO_PREMATURO'] = np.where(
    mortalidade_completa['IDADEANOS'] < 70, 
    'Sim', 
    'Não'
)


Registros duplicados removidos: 0


In [46]:
# 1. Define a lista de colunas que serão importadas
colunas_desejadas = [
    'contador',
    'TIPOBITO',
    'DTOBITO',
    'DTNASC',
    'IDADE',
    'SEXO',
    'RACACOR',
    'ESTCIV',
    'ESC2010',
    'OCUP',
    'CODMUNRES',
    'CODMUNOCOR',
    'CAUSABAS'
]

# 2. Defina os tipos de dados das colunas
tipos_dados = {
    'contador': int,
    'TIPOBITO': str,
    'IDADE': str,       # Mantém como texto para não perder os zeros à esquerda da codificação do DATASUS
    'SEXO': str,
    'RACACOR': str,
    'ESTCIV': str,
    'ESC2010': str,
    'OCUP': str,
    'CODMUNRES': str,
    'CODMUNOCOR': str,
    'CAUSABAS': str     # Garante que códigos CID mistos não gerem alertas
}

# 3. Importa o arquivo CSV de forma otimizada
mortalidade = pd.read_csv(
    '../dados/Mortalidade_Geral_2026.csv',
    sep=';',
    usecols=colunas_desejadas,
    dtype=tipos_dados,
    parse_dates=['DTOBITO', 'DTNASC'],
    date_format='%d/%m/%Y'
)

# 4. Converte DTOBITO e DTNASC (formato: ddmmYYYY) para uma data real do Pandas (formato: YYYY-mm-dd)
mortalidade['DTOBITO'] = pd.to_datetime(mortalidade['DTOBITO'], format='%d%m%Y', errors='coerce')
mortalidade['DTNASC'] = pd.to_datetime(mortalidade['DTNASC'], format='%d%m%Y', errors='coerce')


In [ ]:
# Mostra o tipo de dado das colunas convertidas em data real (formato: YYYY-mm-dd)
print(mortalidade[['DTOBITO', 'DTNASC']].dtypes)

# Mostra o formato das datas convertidas (YYYY-mm-dd)
mortalidade.head()

In [ ]:
mortalidade.info()

In [ ]:
# 1. Separar a IDADE em Tipo (1º dígito) e Quantidade (2º e 3º dígitos)
mortalidade['IDADELIMPA'] = ("000" + mortalidade['IDADE'].astype(str)).str[-3:]

tipo_idade = mortalidade['IDADELIMPA'].str[0]
qtde_idade = pd.to_numeric(mortalidade['IDADELIMPA'].str[1:], errors='coerce')

qtde_idade.loc[qtde_idade == 0] = 1

# 2. Aplicar a regra de conversão para ANOS
mortalidade['IDADEANOS'] = np.select(
    [
        tipo_idade == '0',  # Minutos
        tipo_idade == '1',  # Horas
        tipo_idade == '2',  # Dias
        tipo_idade == '3',  # Meses
        tipo_idade == '4',  # Anos (< 100)
        tipo_idade == '5'   # Anos (>= 100)
    ],
    [
        qtde_idade / (60 * 24 * 365),
        qtde_idade / (24 * 365),
        qtde_idade / 365,
        qtde_idade / 12,
        qtde_idade,
        qtde_idade + 100
    ],
    default=np.nan  # Idades ignoradas ou nulas
)

# 3. Remove a coluna temporária auxiliar
# mortalidade.drop(columns=['IDADELIMPA'], inplace=True)

mortalidade.head()

In [ ]:
# Usando o método .query() (Sintaxe mais limpa)
# mortalidade.query("IDADEANOS < 0.01")

# mortalidade[mortalidade['IDADEANOS'] <= 0.00274] # Idade < que 1 dia
mortalidade[mortalidade['IDADEANOS'] > 0.00274] # Idade > que 1 dia
# mortalidade[(mortalidade['IDADELIMPA'] == '200') & (mortalidade['CODMUNRES'] == '351360')]
# mortalidade[mortalidade['IDADELIMPA'] == '201']
# mortalidade[mortalidade['CODMUNRES'] == '351360']

In [ ]:
mortalidade.describe().round(7)

In [ ]:
mortalidade.info()

In [ ]:
# 1. Define a lista de colunas que serão importadas
colunas_desejadas = [
    'CODMUNRES',
    'MUNICIPIO',
    'UF',
    'POPULAÇÃO'
]

# 2. Defina os tipos de dados das colunas
tipos_dados = {
    'CODMUNRES': str,
    'MUNICIPIO': str,
    'UF': str,
    'POPULAÇÃO': str
}

# 3. Importa o arquivo CSV de forma otimizada
municipios = pd.read_excel(
    'MUNICIPIOS.xlsx',
    usecols=colunas_desejadas,
    dtype=tipos_dados,
    na_values=['null']
)

In [ ]:
# Substitui os nulos da coluna por 0 e converte para inteiro comum
municipios['POPULAÇÃO'] = municipios['POPULAÇÃO'].fillna(0).astype('int64')

# Renomeando a coluna
municipios = municipios.rename(columns={'CODMUNRES': 'CODMUNOCOR'})

municipios.info()

In [ ]:
mort_muni = pd.merge(mortalidade, municipios, on='CODMUNOCOR', how='left')

mort_muni.head()